In [1]:
# ====================================================================================================================
# CUET_Pinnacle@DravidianLangTech 2026
# Ensemble of Indic-Specialized Transformers for Political Multiclass Sentiment Analysis of Tamil X (Twitter) Comments
# ====================================================================================================================

# INSTALL DEPENDENCIES

In [2]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


# IMPORTS

In [3]:
import os
import re
import random
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup,
    set_seed
)
from accelerate import Accelerator
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION

In [4]:
class Config:
    # Data paths (Original Tamil Dataset)
    DATA_DIR = "/kaggle/input/datasets/shuvr09/political-multiclass-sentiment-analysis" 
    TRAIN_FILE = "PS_train.csv"
    DEV_FILE = "PS_dev.csv"
    TEST_FILE = "PS_test.csv" # MAKE SURE THIS POINTS TO YOUR LABELED TEST SET!
    
    # === CORRECTED ENSEMBLE FOR TANGLISH ===
    MODELS = [
        "ai4bharat/IndicBERTv2-MLM-only",      # Best for Indian Languages
        "google/muril-base-cased",             # Best for Transliteration (Tanglish)
        "cardiffnlp/twitter-xlm-roberta-base"  # Best for Twitter structure (Multilingual)
    ]
    MODEL_NAMES_SHORT = ["IndicBERTv2", "MuRIL", "XLM-Twitter"] # For clean tables

    # Ensemble weights [IndicBERT, MuRIL, XLM-T]
    ENSEMBLE_WEIGHTS = [0.35, 0.35, 0.3] 
    
    # Data augmentation settings
    USE_AUGMENTATION = True
    AUGMENTATION_FACTOR = 1.5   
    MIN_SAMPLES_FOR_AUG = 600  
    
    # Train-validation split
    VALIDATION_SPLIT = 0.2
    STRATIFIED_SPLIT = True
    
    # Training hyperparameters
    MAX_LENGTH = 256
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 4
    LEARNING_RATE = 2e-5
    ENCODER_LR = 1e-5
    CLASSIFIER_LR = 3e-5
    EPOCHS = 10
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    
    # Class imbalance handling
    USE_CLASS_WEIGHTS = True
    USE_LABEL_SMOOTHING = True
    LABEL_SMOOTHING = 0.1
    
    # Regularization
    DROPOUT_RATE = 0.3
    
    # Early stopping
    EARLY_STOPPING_PATIENCE = 4
    
    # Misc
    SEED = 42
    FP16 = True
    MAX_GRAD_NORM = 1.0
    OUTPUT_DIR = "./checkpoints"
    SUBMISSION_FILE = "submission.csv"

set_seed(Config.SEED)
random.seed(Config.SEED)
np.random.seed(Config.SEED)

# DATA LOADING (RAW)

In [5]:
print("="*80)
print("LOADING ORIGINAL TANGLISH DATASETS")
print("="*80)

train_df = pd.read_csv(os.path.join(Config.DATA_DIR, Config.TRAIN_FILE))
dev_df = pd.read_csv(os.path.join(Config.DATA_DIR, Config.DEV_FILE))
test_df = pd.read_csv(os.path.join(Config.DATA_DIR, Config.TEST_FILE))

TEST_HAS_LABELS = 'labels' in test_df.columns

print(f"\nOriginal Train size: {len(train_df)}")
print(f"Original Dev size:   {len(dev_df)}")
print(f"Test size:           {len(test_df)}")
print(f"Test set has ground-truth labels: {TEST_HAS_LABELS}")

# Remove empty rows
train_df = train_df[train_df['content'].notna()].reset_index(drop=True)
dev_df = dev_df[dev_df['content'].notna()].reset_index(drop=True)
test_df = test_df[test_df['content'].notna()].reset_index(drop=True)

# Minimal cleanup
train_df['content'] = train_df['content'].astype(str).str.strip()
dev_df['content'] = dev_df['content'].astype(str).str.strip()
test_df['content'] = test_df['content'].astype(str).str.strip()

LOADING ORIGINAL TANGLISH DATASETS

Original Train size: 4352
Original Dev size:   544
Test size:           544
Test set has ground-truth labels: True


# DATA AUGMENTER

In [6]:
class TextAugmenter:
    """Fast augmentation for noisy Tanglish text"""
    @staticmethod
    def random_deletion(text, p=0.1):
        words = text.split()
        if len(words) <= 1: return text
        new_words = [w for w in words if random.random() > p]
        return ' '.join(new_words) if new_words else text
    
    @staticmethod
    def random_swap(text, n=1):
        words = text.split()
        if len(words) < 2: return text
        new_words = words.copy()
        for _ in range(n):
            idx1, idx2 = random.sample(range(len(new_words)), 2)
            new_words[idx1], new_words[idx2] = new_words[idx2], new_words[idx1]
        return ' '.join(new_words)
    
    @staticmethod
    def augment_text(text):
        techniques = [
            lambda x: TextAugmenter.random_deletion(x, p=0.1),
            lambda x: TextAugmenter.random_swap(x, n=1),
        ]
        return random.choice(techniques)(text)

# COMBINE AND SPLIT

In [7]:
print("\n" + "="*80)
print("PREPARING DATA SPLITS")
print("="*80)

# Combine
combined_df = pd.concat([train_df, dev_df], ignore_index=True)

# Label Mapping
unique_labels = sorted(combined_df['labels'].unique().tolist())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
num_labels = len(unique_labels)
combined_df['label_id'] = combined_df['labels'].map(label2id)

# Map labels in test set if they exist
if TEST_HAS_LABELS:
    test_df['label_id'] = test_df['labels'].map(label2id)

print(f"Classes: {unique_labels}")

# Split
print("Splitting data...")
train_df, val_df = train_test_split(
    combined_df,
    test_size=Config.VALIDATION_SPLIT,
    random_state=Config.SEED,
    stratify=combined_df['label_id']
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train split: {len(train_df)}")
print(f"Val split:   {len(val_df)}")


PREPARING DATA SPLITS
Classes: ['Negative', 'Neutral', 'None of the above', 'Opinionated', 'Positive', 'Sarcastic', 'Substantiated']
Splitting data...
Train split: 3916
Val split:   980


# APPLY AUGMENTATION

In [8]:
if Config.USE_AUGMENTATION:
    augmenter = TextAugmenter()
    augmented_samples = []
    
    for label in unique_labels:
        label_df = train_df[train_df['labels'] == label]
        original_count = len(label_df)
        
        if original_count < Config.MIN_SAMPLES_FOR_AUG:
            num_to_generate = min(
                original_count * Config.AUGMENTATION_FACTOR, 
                Config.MIN_SAMPLES_FOR_AUG - original_count
            )
            for _ in range(int(num_to_generate)):
                sample = label_df.sample(n=1).iloc[0]
                aug_text = augmenter.augment_text(sample['content'])
                augmented_samples.append({
                    'content': aug_text,
                    'labels': label,
                    'label_id': label2id[label]
                })
    
    if augmented_samples:
        aug_df = pd.DataFrame(augmented_samples)
        train_df = pd.concat([train_df, aug_df], ignore_index=True)
        train_df = train_df.sample(frac=1, random_state=Config.SEED).reset_index(drop=True)
        print(f"\nFinal Training Size (Post-Augmentation): {len(train_df)}")


Final Training Size (Post-Augmentation): 4717


# CLASS WEIGHTS

In [9]:
if Config.USE_CLASS_WEIGHTS:
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(num_labels),
        y=train_df['label_id'].values
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float32)
else:
    class_weights = None

# DATASET & MODEL

In [10]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts; self.labels = labels; self.tokenizer = tokenizer; self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        item = {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten()}
        if self.labels is not None: item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

class CustomModel(nn.Module):
    def __init__(self, model_name, num_labels, dropout_rate=0.3):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled_output = outputs.pooler_output
        else:
            pooled_output = outputs.last_hidden_state[:, 0, :]
            
        logits = self.classifier(self.dropout(pooled_output))
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(
                weight=class_weights.to(logits.device) if class_weights is not None else None,
                label_smoothing=Config.LABEL_SMOOTHING
            )
            loss = loss_fct(logits, labels)
        return type('Output', (), {'loss': loss, 'logits': logits})()

# TRAINING LOOP

In [11]:
def train_model(model_name, train_df, val_df):
    accelerator = Accelerator(mixed_precision='fp16' if Config.FP16 else 'no', gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = CustomModel(model_name, num_labels, Config.DROPOUT_RATE)
    
    if accelerator.is_main_process: print(f"\nTraining Model: {model_name}")
    
    train_ds = SentimentDataset(train_df['content'].values, train_df['label_id'].values, tokenizer, Config.MAX_LENGTH)
    val_ds = SentimentDataset(val_df['content'].values, val_df['label_id'].values, tokenizer, Config.MAX_LENGTH)
    
    train_dl = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=Config.BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
    
    optim_params = [
        {'params': model.model.parameters(), 'lr': Config.ENCODER_LR},
        {'params': model.classifier.parameters(), 'lr': Config.CLASSIFIER_LR}
    ]
    optimizer = torch.optim.AdamW(optim_params, weight_decay=Config.WEIGHT_DECAY)
    
    num_steps = len(train_dl) * Config.EPOCHS // Config.GRADIENT_ACCUMULATION_STEPS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(num_steps*Config.WARMUP_RATIO), num_steps)
    
    model, optimizer, train_dl, val_dl, scheduler = accelerator.prepare(model, optimizer, train_dl, val_dl, scheduler)
    
    best_f1, best_state, patience = 0.0, None, 0
    
    for epoch in range(Config.EPOCHS):
        model.train()
        for batch in tqdm(train_dl, disable=not accelerator.is_local_main_process, desc=f"Ep {epoch+1}", leave=False):
            with accelerator.accumulate(model):
                out = model(batch['input_ids'], batch['attention_mask'], batch['labels'])
                accelerator.backward(out.loss)
                if accelerator.sync_gradients: accelerator.clip_grad_norm_(model.parameters(), Config.MAX_GRAD_NORM)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
        
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in val_dl:
                out = model(batch['input_ids'], batch['attention_mask'])
                preds.extend(accelerator.gather(out.logits.argmax(-1)).cpu().numpy())
                trues.extend(accelerator.gather(batch['labels']).cpu().numpy())
        
        if accelerator.is_main_process:
            preds, trues = preds[:len(val_ds)], trues[:len(val_ds)]
            f1 = f1_score(trues, preds, average='macro')
            print(f"Epoch {epoch+1} Macro F1: {f1:.4f}")
            
            if f1 > best_f1:
                best_f1, best_state, patience = f1, accelerator.unwrap_model(model).state_dict(), 0
                print("✓ Saved Best")
            else:
                patience += 1
                if patience >= Config.EARLY_STOPPING_PATIENCE: break
        accelerator.wait_for_everyone()
        
    if best_state: accelerator.unwrap_model(model).load_state_dict(best_state)
    return accelerator.unwrap_model(model), tokenizer

# ENSEMBLE EXECUTION

In [12]:
models_list, tokenizers_list = [], []

print("\n" + "="*80)
print("STARTING ENSEMBLE TRAINING")
print("="*80)

for m_name in Config.MODELS:
    m, t = train_model(m_name, train_df, val_df)
    models_list.append(m)
    tokenizers_list.append(t)
    torch.cuda.empty_cache()


STARTING ENSEMBLE TRAINING


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

Epoch 8 Macro F1: 0.3514


Ep 9:   0%|          | 0/590 [00:00<?, ?it/s]

Epoch 9 Macro F1: 0.3532


Ep 10:   0%|          | 0/590 [00:00<?, ?it/s]

Epoch 10 Macro F1: 0.3537


# EVALUATION

In [13]:
def get_logits(model, tokenizer, texts, device):
    model.eval()
    model.to(device)
    dl = DataLoader(SentimentDataset(texts, None, tokenizer, Config.MAX_LENGTH), batch_size=32, shuffle=False)
    logits = []
    with torch.no_grad():
        for b in tqdm(dl, leave=False):
            logits.append(model(b['input_ids'].to(device), b['attention_mask'].to(device)).logits.cpu().numpy())
    return np.vstack(logits)

def calc_quiet_metrics(true_labels, pred_labels):
    """Calculates metrics quietly for the clean summary table"""
    p = precision_score(true_labels, pred_labels, average='macro', zero_division=0)
    r = recall_score(true_labels, pred_labels, average='macro', zero_division=0)
    f = f1_score(true_labels, pred_labels, average='macro', zero_division=0)
    return p, r, f

device = torch.device('cuda')

# ----- 1. VALIDATION EVALUATION -----
print("\n" + "="*80)
print("EXTRACTING VALIDATION METRICS...")
print("="*80)

val_logits_list = []
val_true = val_df['label_id'].values
val_summary = []

for i, (m, t) in enumerate(zip(models_list, tokenizers_list)):
    logits = get_logits(m, t, val_df['content'].values, device)
    val_logits_list.append(logits)
    preds = logits.argmax(1)
    p, r, f = calc_quiet_metrics(val_true, preds)
    val_summary.append({'Model': Config.MODEL_NAMES_SHORT[i], 'P': p, 'R': r, 'MF1': f})

# Weighted Average Validation
ens_val_logits = sum(w * l for w, l in zip(Config.ENSEMBLE_WEIGHTS, val_logits_list))
ens_val_preds = ens_val_logits.argmax(1)
p, r, f = calc_quiet_metrics(val_true, ens_val_preds)
val_summary.append({'Model': 'Weighted Ensemble', 'P': p, 'R': r, 'MF1': f})

print("\n### LATEX TABLE 3 DATA (VALIDATION) ###")
print(pd.DataFrame(val_summary).set_index('Model').to_string(float_format=lambda x: f"{x:.4f}"))

# ----- 2. TEST SET EVALUATION -----
print("\n" + "="*80)
print("EXTRACTING TEST SET METRICS...")
print("="*80)

test_logits_list = []
for i, (m, t) in enumerate(zip(models_list, tokenizers_list)):
    logits = get_logits(m, t, test_df['content'].values, device)
    test_logits_list.append(logits)

ens_test_logits = sum(w * l for w, l in zip(Config.ENSEMBLE_WEIGHTS, test_logits_list))
ens_test_preds = ens_test_logits.argmax(1)

if TEST_HAS_LABELS:
    test_true = test_df['label_id'].values
    test_summary = []

    for i, logits in enumerate(test_logits_list):
        preds = logits.argmax(1)
        p, r, f = calc_quiet_metrics(test_true, preds)
        test_summary.append({'Model': Config.MODEL_NAMES_SHORT[i], 'P': p, 'R': r, 'MF1': f})

    p, r, f = calc_quiet_metrics(test_true, ens_test_preds)
    test_summary.append({'Model': 'Weighted Ensemble', 'P': p, 'R': r, 'MF1': f})

    print("\n### LATEX TABLE 3 DATA (TEST SET) ###")
    print(pd.DataFrame(test_summary).set_index('Model').to_string(float_format=lambda x: f"{x:.4f}"))
    
    # Optional: Detailed final report for the ensemble just to verify classes
    print("\nDetailed Final Classification Report (Test Set):")
    print(classification_report(test_true, ens_test_preds, target_names=unique_labels, digits=4))


EXTRACTING VALIDATION METRICS...


  0%|          | 0/31 [00:00<?, ?it/s]

  0%|          | 0/31 [00:00<?, ?it/s]

  0%|          | 0/31 [00:00<?, ?it/s]


### LATEX TABLE 3 DATA (VALIDATION) ###
                       P      R    MF1
Model                                 
IndicBERTv2       0.3606 0.3760 0.3614
MuRIL             0.2634 0.3109 0.2523
XLM-Twitter       0.3681 0.3625 0.3537
Weighted Ensemble 0.3837 0.3877 0.3775

EXTRACTING TEST SET METRICS...


  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]

  0%|          | 0/17 [00:00<?, ?it/s]


### LATEX TABLE 3 DATA (TEST SET) ###
                       P      R    MF1
Model                                 
IndicBERTv2       0.3569 0.3768 0.3612
MuRIL             0.2886 0.3365 0.2742
XLM-Twitter       0.3544 0.3765 0.3506
Weighted Ensemble 0.3858 0.3975 0.3832

Detailed Final Classification Report (Test Set):
                   precision    recall  f1-score   support

         Negative     0.1791    0.2609    0.2124        46
          Neutral     0.2143    0.1286    0.1607        70
None of the above     0.9600    0.9600    0.9600        25
      Opinionated     0.4632    0.3684    0.4104       171
         Positive     0.3182    0.5600    0.4058        75
        Sarcastic     0.4457    0.3868    0.4141       106
    Substantiated     0.1200    0.1176    0.1188        51

         accuracy                         0.3621       544
        macro avg     0.3858    0.3975    0.3832       544
     weighted avg     0.3744    0.3621    0.3596       544



# CREATE SUBMISSION FILE

In [14]:
print("\nGenerating Final Predictions File...")
test_labels = [id2label[p] for p in ens_test_preds]

sub = pd.DataFrame({'labels': test_labels})
if 'id' in test_df.columns: sub['id'] = test_df['id']
sub.to_csv(Config.SUBMISSION_FILE, index=False)
print(f"Submission saved to {Config.SUBMISSION_FILE}")


Generating Final Predictions File...
Submission saved to submission.csv
